<a href="https://colab.research.google.com/github/Akshatha7710/smart-tea-estate-management-system/blob/soil_quality-analysis_and_predictive_modeling/Soil_Quality_Analysis_and_Predictive_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==========================================================
# STEMS - Smart Tea Estate Management System
# FINAL IMPROVED ML PIPELINE
# Rainfall Forecast -> Prophet
# Wet Days Prediction -> XGBoost
# Soil pH Prediction -> CatBoost
# ==========================================================

# ==========================================================
# 1. Import Libraries
# ==========================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.metrics import mean_absolute_error, r2_score

from prophet import Prophet
import xgboost as xgb
from catboost import CatBoostRegressor

# ==========================================================
# 2. Load Datasets
# ==========================================================
rainfall = pd.read_csv("sample_data/rainfall_data.csv")
soil = pd.read_csv("sample_data/soil_data.csv")

# ==========================================================
# 3. Rainfall Data Preprocessing
# ==========================================================
rainfall["Year"] = rainfall["Year"].str.split("/").str[0].astype(int)

month_map = {
    "January":1,"February":2,"March":3,"April":4,
    "May":5,"June":6,"July":7,"August":8,
    "September":9,"October":10,"November":11,"December":12
}

rainfall["Month"] = rainfall["Month"].map(month_map)

rainfall["Date"] = pd.to_datetime(
    dict(year=rainfall.Year, month=rainfall.Month, day=1)
)

rainfall = rainfall.sort_values("Date").dropna()

# ==========================================================
# 4. Rainfall Forecasting (Prophet)
# ==========================================================
rain_df = rainfall[["Date","Rainfall"]].rename(
    columns={"Date":"ds","Rainfall":"y"}
)

prophet_model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False
)

prophet_model.fit(rain_df)

future = prophet_model.make_future_dataframe(
    periods=12,
    freq="MS"
)

forecast = prophet_model.predict(future)

print("\nRainfall Forecast (Next 12 Months)")
print(forecast[['ds','yhat','yhat_lower','yhat_upper']].tail(12))

# ==========================================================
# 5. Wet Days Prediction (XGBoost)
# ==========================================================
rainfall["Rainfall_lag1"] = rainfall["Rainfall"].shift(1)
rainfall["Rainfall_lag2"] = rainfall["Rainfall"].shift(2)
rainfall["Rainfall_3month_avg"] = rainfall["Rainfall"].rolling(3).mean()

rainfall = rainfall.dropna()

features = [
    "Rainfall",
    "Rainfall_lag1",
    "Rainfall_lag2",
    "Rainfall_3month_avg"
]

X = rainfall[features]
y = rainfall["Wet_days"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    shuffle=False
)

xgb_model = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb_model.fit(X_train, y_train)

wet_pred = xgb_model.predict(X_test)

print("\nWet Days Prediction Performance (XGBoost)")
print("MAE:", mean_absolute_error(y_test, wet_pred))
print("R2:", r2_score(y_test, wet_pred))

# ==========================================================
# 6. Soil Data Cleaning
# ==========================================================
soil.replace("missing", np.nan, inplace=True)

soil["pH"] = pd.to_numeric(soil["pH"], errors="coerce")
soil["C%"] = pd.to_numeric(soil["C%"], errors="coerce")
soil["Year of Planting"] = pd.to_numeric(
    soil["Year of Planting"], errors="coerce"
)

soil = soil.dropna()

# ==========================================================
# 7. Feature Engineering
# ==========================================================
soil["FieldAge"] = 2025 - soil["Year of Planting"]

# Interaction features (important improvement)
soil["C_FieldAge"] = soil["C%"] * soil["FieldAge"]
soil["C_Extent"] = soil["C%"] * soil["Extent (Ha)"]
soil["Age_Extent"] = soil["FieldAge"] * soil["Extent (Ha)"]

# ==========================================================
# 8. Soil pH Prediction (CatBoost)
# ==========================================================
features = [
    "Extent (Ha)",
    "Category",
    "VP/SD",
    "Estate",
    "FieldAge",
    "C%",
    "C_FieldAge",
    "C_Extent",
    "Age_Extent"
]

X = soil[features]
y = soil["pH"]

cat_features = ["Category","VP/SD","Estate"]

X_train, X_test, y_train, y_test = train_test_split(
    X,y,
    test_size=0.2,
    random_state=42
)

cat_model = CatBoostRegressor(
    iterations=600,
    learning_rate=0.03,
    depth=6,
    random_state=42,
    verbose=0
)

cat_model.fit(
    X_train,
    y_train,
    cat_features=cat_features
)

soil_pred = cat_model.predict(X_test)

print("\nSoil pH Prediction Performance (CatBoost)")
print("MAE:", mean_absolute_error(y_test, soil_pred))
print("R2:", r2_score(y_test, soil_pred))

# ==========================================================
# 9. Cross Validation
# ==========================================================
kf = KFold(n_splits=10, shuffle=True, random_state=42)

cv_scores = cross_val_score(
    cat_model,
    X,
    y,
    cv=kf,
    scoring="r2"
)

print("\nCross Validation R2:", cv_scores)
print("Average R2:", cv_scores.mean())

# ==========================================================
# 10. Rainfall Visualization
# ==========================================================
plt.figure(figsize=(10,5))

plt.plot(
    rain_df['ds'],
    rain_df['y'],
    label="Historical Rainfall"
)

plt.plot(
    forecast['ds'],
    forecast['yhat'],
    label="Forecast Rainfall"
)

plt.fill_between(
    forecast['ds'],
    forecast['yhat_lower'],
    forecast['yhat_upper'],
    alpha=0.3
)

plt.title("Rainfall Forecast (Prophet)")
plt.xlabel("Date")
plt.ylabel("Rainfall (mm)")
plt.legend()

plt.show()

ModuleNotFoundError: No module named 'catboost'